# Using Machine Learning to Solve for RSI SMA period
Selection system: two approaches
- Create an algo that performs sentiment and fundamental analysis on crypto, giving it a score then we pick the best performers
- Identify by price action. Price movement 15m, 1hr, 4hr all positive. Then volume.

In [3]:
# Add the project root (one level up from /research) to the path
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from utils.indicators import *
from utils.engine import *

df_orderbook = pd.read_csv("./csv/BTCUSDT_orderbook.csv")
df_trades = pd.read_csv("./csv/BTCUSDT_trades.csv")
# Convert unix timestamp to datetime
df_orderbook['datetime'] = pd.to_datetime(df_orderbook['timestamp'], unit='ms')  # change unit to 's' if in seconds
df_trades['datetime'] = pd.to_datetime(df_trades['timestamp'], unit='ms')

# Set datetime as index
df_orderbook = df_orderbook.set_index('datetime')
df_trades = df_trades.set_index('datetime')

# Aggregate orderbook to 1m (mid-price OHLCV style)
df_orderbook['mid_price'] = (df_orderbook['bid_price'] + df_orderbook['ask_price']) / 2
df_ob_1m = df_orderbook.resample('1min').agg({
    'mid_price': ['first', 'max', 'min', 'last'],
    'bid_price': 'last',
    'ask_price': 'last',
    'bid_amount': 'sum',
    'ask_amount': 'sum',
})
df_ob_1m.columns = ['open', 'high', 'low', 'close', 'bid_price', 'ask_price', 'bid_amount', 'ask_amount']

df_ob_1m['mid_price'] = (df_ob_1m['bid_price'] + df_ob_1m['ask_price']) / 2
df_ob_1m['spread'] = df_ob_1m['ask_price'] - df_ob_1m['bid_price']

df_trades['buy_vol']  = df_trades['amount'].where(df_trades['side'] == 'buy', 0)
df_trades['sell_vol'] = df_trades['amount'].where(df_trades['side'] == 'sell', 0)

# Aggregate trades to 1m
df_trades_1m = df_trades.resample('1min').agg({
    'price': ['first', 'max', 'min', 'last', 'count'],
    'amount': 'sum',
    'buy_vol': ['sum', 'max'],
    'sell_vol': ['sum', 'max']
})

# Flatten multi-level columns
df_trades_1m.columns = ['open', 'high', 'low', 'close', 'num_trades', 'volume', 'agg_buyer', 'largest_buyer', 'agg_seller', 'largest_seller']

df_trades_1m = df_trades_1m.dropna(subset=['close'])

df_price_1m = pd.DataFrame({
    'open': df_ob_1m['open'],
    'high': df_ob_1m['high'],
    'low': df_ob_1m['low'],
    'close': df_ob_1m['close'],
    'largest_buyer': df_trades_1m['largest_buyer'] * 100 / df_trades_1m['volume'],
    'largest_seller': df_trades_1m['largest_seller'] * 100 / df_trades_1m['volume'],
    'OFI_LOB': 100*(df_ob_1m['bid_amount'] - df_ob_1m['ask_amount']) / (df_ob_1m['bid_amount'] + df_ob_1m['ask_amount']),
    'OFI_AGG': 100*(df_trades_1m['agg_buyer'] - df_trades_1m['agg_seller']) / (df_trades_1m['agg_buyer'] + df_trades_1m['agg_seller']),
    'vol_ratio': np.random.rand(len(df_ob_1m)),
    'candle_type': abs(df_ob_1m['close'] - df_ob_1m['open']) / abs(df_ob_1m['high'] - df_ob_1m['low']),
    'candle_size': (df_ob_1m['high'] - df_ob_1m['low']) / (df_ob_1m['high'] - df_ob_1m['low']).rolling(14).mean(),
})
df_price_1m

,open,high,low,close,largest_buyer,largest_seller,OFI_LOB,OFI_AGG,vol_ratio,candle_type,candle_size
datetime,,,,,,,,,,,
2026-06-28 08:59:00,60343.75,60343.75,60337.35,60337.35,14.248252,23.688811,-74.522379,-39.219114,0.084659,1.000000,NaN
2026-06-28 09:00:00,60337.35,60353.45,60334.95,60344.15,7.280285,7.939430,-26.065918,-0.142518,0.764827,0.367568,NaN
2026-06-28 09:01:00,60344.15,60348.75,60311.65,60318.35,4.612850,5.826469,-18.319465,-59.648545,0.116729,0.695418,NaN
2026-06-28 09:02:00,60318.35,60338.15,60318.35,60338.15,7.604817,5.352364,43.894900,8.206958,0.700741,1.000000,NaN
2026-06-28 09:03:00,60338.15,60338.15,60276.75,60286.25,3.312667,3.344345,-19.263835,-13.743947,0.772324,0.845277,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-06-30 19:32:00,58737.05,58763.95,58676.45,58702.15,2.121333,2.754361,7.306696,-12.185402,0.542266,0.398857,1.814815
2026-06-30 19:33:00,58702.15,58747.85,58702.15,58730.65,1.475172,2.510567,14.678857,-20.187012,0.401900,0.623632,0.960661
2026-06-30 19:34:00,58730.65,58755.95,58718.35,58755.95,2.703562,3.827396,-15.112150,-22.699321,0.270655,0.672872,0.797938


# Extracting Target Variable and Advanced Data Manipulation

In [28]:
price_clean = np.asanyarray(df_price_1m['close'], dtype=float)
rsi_2 = get_wilders_rsi(np.diff(price_clean), 2)
rsi_14 = get_wilders_rsi(np.diff(price_clean), 14)
rsi_14_sma = sma(rsi_14, 14)

# 1. Calculate the indicators
price_clean = np.asanyarray(df_price_1m['close'], dtype=float)
diffs = np.diff(price_clean)

# 2. Get the raw RSI arrays
rsi_14_raw = get_wilders_rsi(diffs, 14)
rsi_14_sma_raw = sma(rsi_14_raw, 14)

# 3. Create full-length arrays filled with NaN
rsi_14 = np.full(len(df_price_1m), np.nan)
rsi_14_sma = np.full(len(df_price_1m), np.nan)

rsi_14[-len(rsi_14_raw):] = rsi_14_raw
rsi_14_sma[-len(rsi_14_sma_raw):] = rsi_14_sma_raw

# 5. Assign to DataFrame
df_price_1m['rsi_14'] = rsi_14
df_price_1m['sma_14'] = rsi_14_sma

df_price_1m.to_csv("testing.csv")
# Call it with your variables, explicitly passing labels so the chart is clear
df_price_1m['rsi_above_sma'] = df_price_1m['rsi_14'] > df_price_1m['sma_14']

# 2. Identify where the condition changes (this creates a new "block" for every switch)
df_price_1m['block'] = (df_price_1m['rsi_above_sma'] != df_price_1m['rsi_above_sma'].shift()).cumsum()

# 3. Group by the condition and block to count the duration of each streak
streak_durations = df_price_1m.groupby(['rsi_above_sma', 'block']).size().reset_index(name='duration')

import numpy as np
import pandas as pd
from sklearn.ensemble           import GradientBoostingClassifier

from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection    import TimeSeriesSplit
from sklearn.metrics            import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

df = df_price_1m.reset_index(drop=True).copy()
print(df)
# ─────────────────────────────────────────────────────────────────────────────
# 0 · Setup
# ─────────────────────────────────────────────────────────────────────────────

# ── Recompute block in case the df was rebuilt or the column is stale ────────
df['rsi_above_sma'] = df['rsi_14'] > df['sma_14']
df['block'] = (df['rsi_above_sma'] != df['rsi_above_sma'].shift()).cumsum()

# ── Use transform instead of join — no KeyError, no index alignment issues ──
df['streak_dur'] = df.groupby('block')['block'].transform('count')
is_start        = df['block'] != df['block'].shift()
start_positions = df.index[is_start].tolist()

CLASS_NAMES = ['Short (≤3)', 'Medium (4–10)', 'Long (>10)']

def label(d):
    return 0 if d <= 3 else (1 if d <= 10 else 2)

# ─────────────────────────────────────────────────────────────────────────────
# 1 · Feature extraction — at bar-0 of each new streak only
# ─────────────────────────────────────────────────────────────────────────────

LOOKBACK = 10          # bars of history needed before the crossover

records = []

for i, pos in enumerate(start_positions):
    if pos < LOOKBACK:
        continue

    row  = df.loc[pos]
    hist = df.iloc[pos - LOOKBACK : pos]   # the 10 bars BEFORE crossover (no leakage)
    prev = start_positions[i - 1] if i > 0 else None

    # ── RSI / SMA ──────────────────────────────────────────────────────────
    rsi     = row['rsi_14']
    sma_val = row['sma_14']       # ← safe name, no collision

    # ── Order-flow at the crossover bar ────────────────────────────────────
    ofi_lob = row['OFI_LOB']
    ofi_agg = row['OFI_AGG']

    # ── Order-flow in the run-up (last 5 and 10 bars) ──────────────────────
    ofi_lob_5  = hist.tail(5)['OFI_LOB'].mean()
    ofi_agg_5  = hist.tail(5)['OFI_AGG'].mean()
    ofi_lob_10 = hist['OFI_LOB'].mean()
    ofi_agg_10 = hist['OFI_AGG'].mean()

    # Cumulative order-flow pressure building into the crossover
    ofi_lob_cum = hist['OFI_LOB'].sum()
    ofi_agg_cum = hist['OFI_AGG'].sum()

    # ── Volume imbalance ───────────────────────────────────────────────────
    vr      = row['vol_ratio']
    vr_5    = hist.tail(5)['vol_ratio'].mean()
    vr_10   = hist['vol_ratio'].mean()

    # ── Buyer / seller pressure ────────────────────────────────────────────
    lb       = row['largest_buyer']
    ls       = row['largest_seller']
    bs_ratio = lb / (ls + 1e-9)           # >1 → buyer dominant

    lb_5_mean = hist.tail(5)['largest_buyer'].mean()
    ls_5_mean = hist.tail(5)['largest_seller'].mean()

    # ── Candle character at crossover ──────────────────────────────────────
    c_type = row['candle_type']
    c_size = row['candle_size']

    c_size_5_mean = hist.tail(5)['candle_size'].mean()
    c_type_5_mean = hist.tail(5)['candle_type'].mean()   # avg sentiment

    # ── RSI dynamics ───────────────────────────────────────────────────────
    rsi_sma_dist = rsi - sma_val
    rsi_vs_50    = rsi - 50
    rsi_d3       = rsi - df.at[pos - 3, 'rsi_14']
    rsi_d5       = rsi - df.at[pos - 5, 'rsi_14']

    rec = {
        # RSI / SMA
        'rsi_sma_dist':   rsi_sma_dist,

        # Order flow — spot

        'ofi_agg':        ofi_agg,
        'ofi_alignment':  int(np.sign(ofi_lob) == np.sign(ofi_agg)),  # both agree?


        # Volume

        # Candle
        'candle_type':    c_type,

        'target':         label(row['streak_dur']),
        'streak_dur':     row['streak_dur'],
    }

    records.append(rec)

feat_df = pd.DataFrame(records).dropna()

print(f"Usable streaks: {len(feat_df)}")
print("\nClass distribution:")
print(feat_df['target'].value_counts().sort_index()
      .rename(index=dict(enumerate(CLASS_NAMES))))
# ─────────────────────────────────────────────────────────────────────────────
# 2 · Train / test split  (chronological — never shuffle)
# ─────────────────────────────────────────────────────────────────────────────

META  = ['target', 'streak_dur']
FEATS = [c for c in feat_df.columns if c not in META]

X = feat_df[FEATS].values
y = feat_df['target'].values

cut        = int(len(feat_df) * 0.80)
X_tr, X_te = X[:cut], X[cut:]
y_tr, y_te = y[:cut], y[cut:]

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Sample weights are handled automatically by class_weight='balanced' in LogisticRegression
# sw_tr = compute_sample_weight('balanced', y_tr)

# ─────────────────────────────────────────────────────────────────────────────
# 3 · Train (L1 Regularized Logistic Regression / "Lasso" Classification)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression


# ── Updated Model Initialization ─────────────────────────────────────────────
# solver='saga' handles native multiclass with L1 penalty.
# max_iter=5000 ensures it converges properly.
clf = LogisticRegression(
    penalty='l1',
    solver='saga',      # <-- Changed from liblinear
    max_iter=5000,      # <-- Added to ensure convergence
    C=0.05, 
    class_weight='balanced',
    random_state=42
)

clf.fit(X_tr_s, y_tr)


          open      high       low     close  largest_buyer  largest_seller  \
0     60343.75  60343.75  60337.35  60337.35      14.248252       23.688811   
1     60337.35  60353.45  60334.95  60344.15       7.280285        7.939430   
2     60344.15  60348.75  60311.65  60318.35       4.612850        5.826469   
3     60318.35  60338.15  60318.35  60338.15       7.604817        5.352364   
4     60338.15  60338.15  60276.75  60286.25       3.312667        3.344345   
...        ...       ...       ...       ...            ...             ...   
3513  58737.05  58763.95  58676.45  58702.15       2.121333        2.754361   
3514  58702.15  58747.85  58702.15  58730.65       1.475172        2.510567   
3515  58730.65  58755.95  58718.35  58755.95       2.703562        3.827396   
3516  58755.95  58794.55  58755.95  58759.05       1.786655        3.659964   
3517  58759.05  58759.05  58723.15  58755.85       3.533569        3.344789   

        OFI_LOB    OFI_AGG  vol_ratio  candle_type 

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.05
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`T

# Machine Learning Test Fire and Finding Meaningful Variables
1. RSI-SMA Dist: Meaningful as it represents distance from SMA directly, potentially telling us how strong the trend is.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4 · Evaluate
# ─────────────────────────────────────────────────────────────────────────────

present_classes = sorted(np.unique(np.concatenate([y_tr, y_te])))
present_names   = [CLASS_NAMES[c] for c in present_classes]

y_pred = clf.predict(X_te_s)
proba  = clf.predict_proba(X_te_s)

print("\n── Classification Report (80/20 Split) ───────────────────────────")
print(classification_report(y_te, y_pred,
                             labels      = present_classes,
                             target_names= present_names))

cm = confusion_matrix(y_te, y_pred, labels=present_classes)
print("── Confusion Matrix ──────────────────────────────────────────────")
print(pd.DataFrame(cm,
      index   = [f"True  {n}" for n in present_names],
      columns = [f"Pred  {n}" for n in present_names]))

# ── Leave-One-Out Cross Validation (Best for N < 100) ───────────────────
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

loo = LeaveOneOut()
loo_y_true = []
loo_y_pred = []

# Scale the WHOLE dataset for LOOCV to simulate training on n-1
X_all_s = scaler.fit_transform(X)

for tr_idx, val_idx in loo.split(X_all_s):
    X_train_cv, X_test_cv = X_all_s[tr_idx], X_all_s[val_idx]
    y_train_cv, y_test_cv = y[tr_idx], y[val_idx]
    
    m = LogisticRegression(penalty='l1', solver='saga', max_iter=5000, C=0.05, class_weight='balanced', random_state=42)
    m.fit(X_train_cv, y_train_cv)
    
    loo_y_pred.append(m.predict(X_test_cv)[0])
    loo_y_true.append(y_test_cv[0])

loo_f1 = f1_score(loo_y_true, loo_y_pred, labels=present_classes, average='macro')
loo_acc = accuracy_score(loo_y_true, loo_y_pred)

print(f"\n── Leave-One-Out CV Results (Entire Dataset) ─────────────────────")
print(f"LOOCV Accuracy:   {loo_acc:.3f}")
print(f"LOOCV Macro-F1:   {loo_f1:.3f}")


# ─────────────────────────────────────────────────────────────────────────────
# 5 · Feature Importance (Coefficients)
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# For multiclass Logistic Regression, clf.coef_ has shape (n_classes, n_features)
# We take the mean of the absolute coefficients across all classes to see overall importance
mean_abs_coefs = np.mean(np.abs(clf.coef_), axis=0)

feature_importances = pd.Series(mean_abs_coefs, index=FEATS)

# Features with a coefficient exactly equal to 0.0 were removed by the L1/Lasso penalty
print("\n── Features Dropped by L1 Penalty ────────────────────────────────")
dropped_feats = feature_importances[feature_importances == 0].index.tolist()
if dropped_feats:
    for feat in dropped_feats:
        print(f" - {feat}")
else:
    print("None. (Try lowering the 'C' parameter in LogisticRegression to drop more noise).")

# Plot top features
top_15 = feature_importances[feature_importances > 0].nlargest(15)

if not top_15.empty:
    plt.figure(figsize=(10, 6))
    top_15.plot(kind='barh', color='coral')
    plt.title("Top Feature Importances (Absolute L1 Coefficients)")
    plt.xlabel("Mean Absolute Coefficient Size")
    plt.gca().invert_yaxis()
    plt.show()
else:
    print("Warning: All feature coefficients were zeroed out by Lasso. Your features may be pure noise.")
# ─────────────────────────────────────────────────────────────────────────────
# 6 · Mutual Information
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.feature_selection import mutual_info_classif

X_mi = feat_df.drop(columns=['target', 'streak_dur'])
y_mi = feat_df['target']

mi_scores = mutual_info_classif(X_mi, y_mi, discrete_features=False, random_state=42)
mi_series = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)

print("\n── Mutual Information Scores ─────────────────────────────────────")
print(mi_series[mi_series > 0])


── Classification Report (80/20 Split) ───────────────────────────
               precision    recall  f1-score   support

   Short (≤3)       0.00      0.00      0.00        13
Medium (4–10)       0.00      0.00      0.00         8
   Long (>10)       0.34      1.00      0.51        11

     accuracy                           0.34        32
    macro avg       0.11      0.33      0.17        32
 weighted avg       0.12      0.34      0.18        32

── Confusion Matrix ──────────────────────────────────────────────
                     Pred  Short (≤3)  Pred  Medium (4–10)  Pred  Long (>10)
True  Short (≤3)                    0                    0                13
True  Medium (4–10)                 0                    0                 8
True  Long (>10)                    0                    0                11

── Leave-One-Out CV Results (Entire Dataset) ─────────────────────
LOOCV Accuracy:   0.335
LOOCV Macro-F1:   0.306

── Features Dropped by L1 Penalty ──────────────────

# Conclusion
Model performs terribly and cannot be relied on. RSI-SMA dist variable hasn't been stress tested among different regimes, but its expected to underperform random variable ('vol ratio')
- In a favorable market where RSI_SMA_dist performs well, the model performs decently (not good enough)
![Feature importance + Classification report](./image/img_5.png)
![Mutual Information](./image/img_6.png)
